In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score  # splitting data, cross-validation
from sklearn.ensemble import GradientBoostingClassifier                # BDT algorithm
from sklearn.metrics import (
    roc_curve, roc_auc_score, confusion_matrix,                        # tools to measure how good the model is
    classification_report, ConfusionMatrixDisplay
)

In [ ]:
df_ele = pd.read_csv(ELECTRON_CSV)
df_pim = pd.read_csv(PION_CSV)

df_ele["label"] = 1  # electron
df_pim["label"] = 0  # pion

df = pd.concat([df_ele, df_pim], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Loaded {len(df)} events "
      f"({df['label'].sum()} electrons, {(df['label']==0).sum()} pions)")
df.head()

In [ ]:
FEATURE_COLUMNS = [
    "npe_HTCC",
    "Edep_ECin",
    "Edep_ECout",
    "Edep_Pcal",
    "E",# momentum >4 and <4
]

missing = [c for c in FEATURE_COLUMNS if c not in df.columns]
if missing:
    raise ValueError(
        f"Missing columns: {missing}\nAvailable: {list(df.columns)}\n"
        f"Update FEATURE_COLUMNS / column names above to match your CSV."
    )

X = df[FEATURE_COLUMNS]
y = df["label"]

print(f"Using features: {FEATURE_COLUMNS}")

# Train / Test Split

75% train, 25% held-out test - the model is never evaluated on data it trained on. `stratify=y` keeps the electron/pion ratio consistent in both halves.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}   Test: {len(X_test)}")

# 4. Train the BDT

200 shallow trees, each correcting the previous ones' mistakes. Followed by 5-fold cross-validation on the training set only, to confirm performance isn't a fluke of one particular split.

In [ ]:
bdt = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
)
bdt.fit(X_train, y_train)

cv_scores = cross_val_score(bdt, X_train, y_train, cv=5, scoring="roc_auc")
print(f"5-fold CV ROC AUC: {cv_scores.mean():.4f} \u00b1 {cv_scores.std():.4f}")

In [ ]:
y_proba = bdt.predict_proba(X_test)[:, 1]
y_pred = bdt.predict(X_test)

print("--- Classification report ---")
print(classification_report(y_test, y_pred, target_names=["pion", "electron"]))

auc = roc_auc_score(y_test, y_proba)
print(f"Test ROC AUC: {auc:.4f}")

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
purity = tp / (tp + fp)
efficiency = tp / (tp + fn)
print(f"Purity:     {purity:.4f}")
print(f"Efficiency: {efficiency:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Confusion matrix
ConfusionMatrixDisplay(cm, display_labels=["pion", "electron"]).plot(
    ax=axes[0], colorbar=False, cmap="Blues"
)
axes[0].set_title("Confusion Matrix")

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, label=f"AUC = {auc:.3f}")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend()

# Feature importance
importances = pd.Series(bdt.feature_importances_, index=FEATURE_COLUMNS)
importances = importances.sort_values(ascending=True)
axes[2].barh(importances.index, importances.values)
axes[2].set_title("BDT Feature Importance")
axes[2].set_xlabel("Importance")

plt.tight_layout()
plt.savefig("bdt_course_evaluation.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: bdt_course_evaluation.png")